# Lang2Logic 

In [2]:
%%capture
!pip install openai nltk lark sympy --quiet

In [4]:
import os, sys

# --- API Key ---
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
    print("Key loaded from Kaggle Secrets.")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."   # fallback
    print("Key loaded from fallback.")

# --- Import pipeline ---
sys.path.insert(0, "/kaggle/input/datasets/remiiyoo/lang2logic-fixed-price")
from lang2logic import Lang2Logic

print("Lang2Logic ready.")

Key loaded from Kaggle Secrets.
Lang2Logic ready.


---
## Example 1 — Circus (paper Figure 3)

Ví dụ gốc từ paper. 

In [5]:
circus = """
The circus has a ferris wheel or the circus has a rollercoaster.
The circus does not have a carousel if and only if the circus has a ferris wheel
and the circus has a rollercoaster.
If the circus does not have a carousel, then the circus has a trapese.
The circus does not have a trapese and the circus has a rollercoaster.
"""

p1 = Lang2Logic(model="o4-mini")
r1 = p1.convert(circus, verbose=True)
print(f"API cost: ${p1.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 4 câu

  Sentence #1: The circus has a ferris wheel or the circus has a rollercoaster.
  Model Output: Or(F, R)
  CNF Expression: F | R

  Sentence #2: The circus does not have a carousel if and only if the circus has a ferris wheel
and the circus has a rollercoaster.
  Model Output: Equivalent(Not(C), And(F, R))
  CNF Expression: (C | F) & (C | R) & (~C | ~F | ~R)

  Sentence #3: If the circus does not have a carousel, then the circus has a trapese.
  Model Output: Implies(Not(C), T)
  CNF Expression: C | T

  Sentence #4: The circus does not have a trapese and the circus has a rollercoaster.
  Model Output: And(Not(T), R)
  CNF Expression: R & ~T


  Final CNF (Simplified):
  C & R & ~F & ~T
  Variables (4): ['F', 'R', 'C', 'T']
  Meanings: {'F': 'The circus has a ferris wheel', 'R': 'The circus has a rollercoaster', 'C': 'The circus has a carousel', 'T': 'The circus has a trapeze'}
  Clauses (approx): 4
  API tokens used: 2666


API cost: $0.002933 USD


---
## Example 2 — Implication chain 

Ba câu `Implies` nối tiếp.

In [6]:
rain = """
If it is raining then you need an umbrella.
If you need an umbrella then you stay dry.
It is raining.
"""

p2 = Lang2Logic(model="o4-mini")
r2 = p2.convert(rain, verbose=True)
print(f"API cost: ${p2.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 3 câu

  Sentence #1: If it is raining then you need an umbrella.
  Model Output: Implies(A, B)
  CNF Expression: B | ~A

  Sentence #2: If you need an umbrella then you stay dry.
  Model Output: Implies(B, C)
  CNF Expression: C | ~B

  Sentence #3: It is raining.
  Model Output: A
  CNF Expression: A


  Final CNF (Simplified):
  A & B & C
  Variables (3): ['A', 'B', 'C']
  Meanings: {'A': 'It is raining', 'B': 'You need an umbrella', 'C': 'You stay dry'}
  Clauses (approx): 3
  API tokens used: 1743


API cost: $0.001917 USD


---
## Example 3 — Contradiction → UNSAT

Hai literal phủ nhau kết hợp với disjunction. Kết quả phải là `False`.

In [7]:
contradiction = """
Alice is at home or Alice is at work.
Alice is not at home.
Alice is not at work.
"""

p3 = Lang2Logic(model="o4-mini")
r3 = p3.convert(contradiction, verbose=True)
print(f"API cost: ${p3.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 3 câu

  Sentence #1: Alice is at home or Alice is at work.
  Model Output: Or(H, W)
  CNF Expression: H | W

  Sentence #2: Alice is not at home.
  Model Output: Not(H)
  CNF Expression: ~H

  Sentence #3: Alice is not at work.
  Model Output: Not(W)
  CNF Expression: ~W


  Final CNF (Simplified):
  False
  Variables (2): ['H', 'W']
  Meanings: {'H': 'Alice is at home', 'W': 'Alice is at work'}
  Clauses (approx): 1
  API tokens used: 1730


API cost: $0.001903 USD


---
## Example 4 — Exclusive-or

Grammar của Lang2Logic không hỗ trợ `Xor()` . Model phải tự encode: `And(Or(A,B), Not(And(A,B)))`.

In [8]:
xor = """
Either the alarm rings or the backup system fires, but not both.
The backup system fires.
"""

p4 = Lang2Logic(model="o4-mini")
r4 = p4.convert(xor, verbose=True)
print(f"API cost: ${p4.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 2 câu

  Sentence #1: Either the alarm rings or the backup system fires, but not both.
  Model Output: And(Or(A, B), Not(And(A, B)))
  CNF Expression: (A | B) & (~A | ~B)

  Sentence #2: The backup system fires.
  Model Output: B
  CNF Expression: B


  Final CNF (Simplified):
  B & ~A
  Variables (2): ['A', 'B']
  Meanings: {'A': 'the alarm rings', 'B': 'the backup system fires'}
  Clauses (approx): 2
  API tokens used: 1150


API cost: $0.001265 USD


---
## Example 5 — Biconditional + unit facts (SAT)

`Equivalent` kết hợp với hai unit clause. Unit propagation phải force `P = True`.

In [9]:
biconditional = """
A student passes the exam if and only if they study hard or they get tutoring.
The student does not study hard.
The student gets tutoring.
"""

p5 = Lang2Logic(model="o4-mini")
r5 = p5.convert(biconditional, verbose=True)
print(f"API cost: ${p5.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 3 câu

  Sentence #1: A student passes the exam if and only if they study hard or they get tutoring.
  Model Output: Equivalent(P, Or(S, T))
  CNF Expression: (P | ~S) & (P | ~T) & (S | T | ~P)

  Sentence #2: The student does not study hard.
  Model Output: Not(S)
  CNF Expression: ~S

  Sentence #3: The student gets tutoring.
  Model Output: T
  CNF Expression: T


  Final CNF (Simplified):
  P & T & ~S
  Variables (3): ['P', 'S', 'T']
  Meanings: {'P': 'A student passes the exam', 'S': 'They study hard', 'T': 'They get tutoring'}
  Clauses (approx): 3
  API tokens used: 1947


API cost: $0.002142 USD


---
## Example 6 — Smart home security (comprehensive)

Ví dụ toàn diện hơn: 5 câu, 5 biến, dùng đủ `Equivalent` / `Implies` / `Not` / atom.  
Kiểm tra **variable reuse**.

In [10]:
security = """
The security alarm is triggered if and only if there is motion and the door is unlocked.
If the alarm is triggered then the security company is notified.
If the security company is notified then the police are called.
The door is unlocked.
There is no motion detected.
"""

p6 = Lang2Logic(model="o4-mini")
r6 = p6.convert(security, verbose=True)
print(f"API cost: ${p6.estimate_cost_usd():.6f} USD")


[Lang2Logic] Input: 5 câu

  Sentence #1: The security alarm is triggered if and only if there is motion and the door is unlocked.
  Model Output: Equivalent(T, And(M, U))
  CNF Expression: (M | ~T) & (U | ~T) & (T | ~M | ~U)

  Sentence #2: If the alarm is triggered then the security company is notified.
  Model Output: Implies(T, N)
  CNF Expression: N | ~T

  Sentence #3: If the security company is notified then the police are called.
  Model Output: Implies(N, P)
  CNF Expression: P | ~N

  Sentence #4: The door is unlocked.
  Model Output: U
  CNF Expression: U

  Sentence #5: There is no motion detected.
  Model Output: Not(M)
  CNF Expression: ~M


  Final CNF (Simplified):
  U & ~M & ~T & (P | ~N)
  Variables (5): ['T', 'M', 'U', 'N', 'P']
  Meanings: {'T': 'The security alarm is triggered', 'M': 'There is motion', 'U': 'The door is unlocked', 'N': 'The security company is notified', 'P': 'The police are called'}
  Clauses (approx): 4
  API tokens used: 3024


API cost: $0.003

---
## Summary table

In [11]:
import pandas as pd
from sympy.logic.boolalg import And as SympyAnd
from sympy.logic.inference import satisfiable

all_results = [
    ("1", "Circus",           r1),
    ("2", "Rain chain",       r2),
    ("3", "Contradiction",    r3),
    ("4", "XOR",              r4),
    ("5", "Biconditional",    r5),
    ("6", "Smart home",       r6),
]

rows = []
for num, name, r in all_results:
    cnf  = r.get("final_cnf", "")
    sat  = "UNSAT" if str(cnf) == "False" else "SAT"
    rows.append({
        "#"         : num,
        "Example"   : name,
        "Sentences" : len(r.get("sentences", [])),
        "Vars"      : len(r.get("variables", [])),
        "Clauses"   : r.get("n_clauses", 0),
        "Final CNF" : cnf,
        "Result"    : sat,
    })

df = pd.DataFrame(rows)

def color_result(val):
    if val == "SAT":   return "background-color:#27ae60;color:white;font-weight:bold"
    if val == "UNSAT": return "background-color:#c0392b;color:white;font-weight:bold"
    return ""

display(df.style.applymap(color_result, subset=["Result"]).hide(axis="index"))

/tmp/ipykernel_57/3985565875.py:35: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(df.style.applymap(color_result, subset=["Result"]).hide(axis="index"))


#,Example,Sentences,Vars,Clauses,Final CNF,Result
1,Circus,4,4,4,C & R & ~F & ~T,SAT
2,Rain chain,3,3,3,A & B & C,SAT
3,Contradiction,3,2,1,False,UNSAT
4,XOR,2,2,2,B & ~A,SAT
5,Biconditional,3,3,3,P & T & ~S,SAT
6,Smart home,5,5,4,U & ~M & ~T & (P | ~N),SAT
